# DataMend — rainfall anomaly model: train & inspect (Kaggle / Colab / any machine)

Upload this notebook to Kaggle, attach your RF25 dataset (**Add Input**), turn on a **GPU**, then *Run All*.
It clones the repo, explains and visualizes the data, trains, evaluates, and plots results. All settings are in the next cell.

**What is trained.** Models that look at a window of daily rainfall at one grid cell and flag each day as plausible or anomalous.
RF25 has no human QC labels, so the training data is *clean RF25 windows with synthetic errors injected*
(spike, stuck, scale, phantom_rain, zeroed, repeat). Models output a zero-inflated Gamma distribution for the clean
value (`nll` score), a supervised anomaly logit (`head`) and a residual (`resid`); scores are calibrated on clean validation windows.

**Honest scope.** Scores on injected anomalies measure detection of *synthetic* errors only. Transfer to real IMD errors is a hypothesis.

## 1. Settings

In [ ]:
# ---- edit these ----
REPO_URL   = "https://github.com/NL0711/datamend.git"
BRANCH     = "ml-training"
MODE       = "single"      # "single" = one model | "ensemble" = lstm_baseline + base + scalenet fused + ablation
PRESET     = "base"        # single mode: smoke | lstm_baseline | base | large | xlarge | scalenet | scalenet_single | scalenet_large | multiscale | multiscale_large
ENS_PRESET = "ensemble"    # ensemble mode: smoke | ensemble | ensemble_large
EPOCHS     = 10
MAX_TRAIN_YEARS = None     # e.g. 10 for a quick check (uses the most recent N train years); None = all
SEEDS      = [0, 1, 2]     # ensemble mode: every seed is reported (no best-of-N)
FUSION     = "mean_rank"   # ensemble: mean_rank | max | prob_avg | owa
DATA_DIR   = None          # None = auto-detect under /kaggle/input; or set a path / DATAMEND_DATA_DIR
RUN_NAME   = "run1"
GITHUB_TOKEN = None        # only if the repo is private (use Kaggle Secrets, never paste a token here)

## 2. Get the code
Clones the repo into the working directory (or pulls if it already exists) and installs the few extra dependencies.
`torch` is preinstalled on Kaggle/Colab, so it is not reinstalled.

In [ ]:
import os, subprocess, sys, pathlib

def sh(cmd, cwd=None):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-2000:])
    if r.returncode: raise RuntimeError(f"failed: {cmd}")

# If this notebook already sits inside the repo (local run), use it; otherwise clone.
here = pathlib.Path.cwd()
local = next((p for p in [here, *here.parents] if (p / "ml-service" / "training").is_dir()), None)
if local:
    REPO = local
else:
    root = pathlib.Path("/kaggle/working") if pathlib.Path("/kaggle/working").is_dir() else here
    REPO = root / "datamend"
    url = REPO_URL if not GITHUB_TOKEN else REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
    if REPO.exists():
        sh(f"git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}", cwd=REPO)
    else:
        sh(f"git clone -b {BRANCH} {url} {REPO}")
ML = REPO / "ml-service"
os.chdir(ML)
sys.path.insert(0, str(ML))
sh(f"{sys.executable} -m pip install -q -r requirements-train.txt")
sh("git log --oneline -1", cwd=REPO)

## 3. Environment check
Confirms the GPU, locates the RF25 `.nc` files (`training/paths.py` searches `--data-dir`, `$DATAMEND_DATA_DIR`, `../data`, then `/kaggle/input/**`),
and sets the output directory (`/kaggle/working` on Kaggle, so results show in the Output tab).

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from training.paths import resolve_data_dir, resolve_out_dir

print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only (training will be slow)")
data_dir = resolve_data_dir(DATA_DIR)
OUT = resolve_out_dir(f"artifacts/{RUN_NAME}")
files = sorted(data_dir.glob("RF25_ind*_rfp25.nc"))
print("data dir:", data_dir, "| files:", len(files), "|", files[0].name, "...", files[-1].name)
print("output dir:", OUT)

## 4. The data
`RainfallArchive` is a lazy view over one NetCDF file per year, giving float32 cubes of shape *(days, lat, lon)* in mm;
off-land cells are NaN. Missing years are reported and never interpolated.

In [ ]:
from training.data import RainfallArchive
arch = RainfallArchive(data_dir)
print(f"years {arch.years[0]}-{arch.years[-1]} ({len(arch.years)} files) | missing years: {arch.missing_years or 'none'}")
print(f"grid {arch.land_mask.shape} (lat x lon) | land cells: {int(arch.land_mask.sum())} | cell size ~27.8 km (0.25 deg)")

In [ ]:
YEAR = arch.years[len(arch.years)//2]
cube = arch.load(YEAR)                       # (days, H, W) mm
extent = [arch.lon.min(), arch.lon.max(), arch.lat.min(), arch.lat.max()]
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
im = ax[0].imshow(np.nanmean(cube, 0), origin="lower", extent=extent, cmap="Blues"); ax[0].set_title(f"{YEAR}: mean daily rain (mm/day)"); plt.colorbar(im, ax=ax[0])
im = ax[1].imshow(np.nansum(cube > 0.1, 0) / cube.shape[0], origin="lower", extent=extent, cmap="viridis"); ax[1].set_title(f"{YEAR}: fraction of wet days (>0.1 mm)"); plt.colorbar(im, ax=ax[1])
im = ax[2].imshow(np.nanmax(cube, 0), origin="lower", extent=extent, cmap="magma"); ax[2].set_title(f"{YEAR}: wettest day (mm)"); plt.colorbar(im, ax=ax[2])
for a in ax: a.set_xlabel("lon"); a.set_ylabel("lat")
plt.tight_layout(); plt.show()

In [ ]:
# Why a zero-inflated Gamma? Most days are dry and wet-day amounts are heavy-tailed.
wet = cube[np.isfinite(cube)]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].bar(["dry (<=0.1 mm)", "wet (>0.1 mm)"], [(wet <= 0.1).mean(), (wet > 0.1).mean()]); ax[0].set_title("share of land cell-days")
ax[1].hist(wet[wet > 0.1], bins=np.logspace(-1, np.log10(max(wet.max(), 1.1)), 60)); ax[1].set_xscale("log"); ax[1].set_yscale("log")
ax[1].set_title("wet-day amount (log-log): heavy tail"); ax[1].set_xlabel("mm/day")
plt.tight_layout(); plt.show()

# A single cell's year.
land = np.argwhere(arch.land_mask)
r, c = land[len(land) // 2]
fig, ax = plt.subplots(figsize=(13, 3.2)); ax.bar(range(cube.shape[0]), np.nan_to_num(cube[:, r, c]), width=1.0)
ax.set_title(f"cell ({arch.lat[r]:.2f}N, {arch.lon[c]:.2f}E), {YEAR}: daily rainfall (mm)"); ax.set_xlabel("day of year"); plt.show()

## 5. Chronological split and what is learned from train years only
`split_years`: the latest 4 years are **test**, the 3 before are **val**, the rest **train**. No random splitting, so no temporal leakage.
Transform (`log1p(mm)/scale`) and per-cell day-of-year climatology are fit on train years only (`training/leakage.py` asserts this).

In [ ]:
from training.splits import split_years
sp = split_years(arch.years)
print(f"train {sp.train[0]}-{sp.train[-1]} ({len(sp.train)} yrs) | val {sp.val} | test {sp.test}")
fig, ax = plt.subplots(figsize=(12, 1.2))
for yrs, col, name in [(sp.train, "tab:blue", "train"), (sp.val, "tab:orange", "val"), (sp.test, "tab:red", "test")]:
    ax.scatter(yrs, [0] * len(yrs), c=col, s=25, label=name)
ax.set_yticks([]); ax.legend(ncol=3, loc="upper left"); plt.show()

## 6. The synthetic anomalies
Training windows are corrupted with one of six error kinds (labels mark only days whose value actually changed). Examples on a real wet window:

In [ ]:
from training.injection import inject, KINDS
rng = np.random.default_rng(3)
W = 60
series = np.nan_to_num(cube[:, r, c])
t0 = int(np.argmax(np.convolve(series, np.ones(W), "valid")))      # wettest 60-day stretch
base = series[t0:t0 + W][None].astype(np.float32)
fig, axes = plt.subplots(len(KINDS), 1, figsize=(12, 2.0 * len(KINDS)), sharex=True)
for ax, kind in zip(axes, KINDS):
    out, lab = base, np.zeros_like(base, bool)
    for _ in range(300):   # re-draw until this kind actually fires
        out, lab, ki = inject(base, rng, rate=1.0, kinds=(kind,))
        if lab.any(): break
    ax.bar(range(W), base[0], color="lightgray", width=1.0, label="clean")
    ax.bar(np.where(lab[0])[0], out[0][lab[0]], color="crimson", width=1.0, label="injected")
    ax.set_ylabel(kind, rotation=0, ha="right")
axes[0].legend(ncol=2); axes[-1].set_xlabel("day in window"); plt.tight_layout(); plt.show()

## 7. Training
**Algorithm:** each step samples random (cell, year, start-day) windows from train years, corrupts about half, and minimises
`ZIG negative log-likelihood (clean target) + weighted BCE (injected labels)` with AdamW, warmup + cosine LR, gradient clipping 1.0,
AMP on GPU and EMA weights for validation/inference. Epochs are fixed counts of random-window steps, not passes over the data.
`MODE="single"` trains one model; `MODE="ensemble"` trains `lstm_baseline`, `base` and `scalenet`, fuses them label-free and runs the single- vs multi-scale ablation.

In [ ]:
import json, time
from dataclasses import asdict
from training import artifact
from training.pipeline import prepare, fit_member
from training.presets import make_config
from training.evaluate import evaluate

t_start = time.time()
if MODE == "single":
    cfg = make_config(PRESET)
    prep = prepare(data_dir, max_train_years=MAX_TRAIN_YEARS, coarse_blocks=cfg.coarse_blocks)
    m = fit_member(prep, cfg, EPOCHS)
    res = {"val": evaluate(m.trainer, m.val_s, m.cal, 4096, seed=11),
           "test": evaluate(m.trainer, m.test_s, m.cal, 4096, seed=0), "history": m.history, "config": cfg.to_dict()}
    artifact.save(artifact.Artifact(m.trainer, m.cal, {
        "train_years": prep.split.train, "val_years": prep.split.val, "test_years": prep.split.test,
        "history": m.history, "sources": [asdict(s) for s in m.train_s.specs],
        "val_metrics": res["val"], "test_metrics": res["test"]}), OUT)
    print(f"saved model to {OUT}")
else:
    cmd = [sys.executable, "-m", "training", "ensemble", "--preset", ENS_PRESET, "--fusion", FUSION,
           "--epochs", str(EPOCHS), "--seeds", *map(str, SEEDS), "--out", OUT, "--data-dir", str(data_dir)]
    if MAX_TRAIN_YEARS: cmd += ["--max-train-years", str(MAX_TRAIN_YEARS)]
    subprocess.run(cmd, check=True)
print(f"done in {(time.time() - t_start) / 60:.1f} min")

## 8. Results

In [ ]:
if MODE == "single":
    h = res["history"]
    fig, ax = plt.subplots(1, 3, figsize=(17, 3.8))
    ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["val_loss"], label="val (clean windows)"); ax[0].set_title("loss per epoch"); ax[0].set_xlabel("epoch"); ax[0].legend()
    sc = res["test"]["scores"]
    ax[1].barh(list(sc), [v["roc_auc"] for v in sc.values()]); ax[1].set_xlim(0.5, 1); ax[1].set_title("test ROC-AUC by score stream\n(baseline = no-model climatology/neighbour disagreement)")
    pk = res["test"]["per_kind_roc_auc"]
    ax[2].barh(list(pk), [v["roc_auc"] for v in pk.values()], color=["tab:blue" if v["seen"] else "tab:red" for v in pk.values()])
    ax[2].set_xlim(0.5, 1); ax[2].set_title("test ROC-AUC by anomaly kind (red = held out)")
    plt.tight_layout(); plt.show()
    p = res["test"]["primary"]
    print(f"test fused: ROC-AUC {p['roc_auc']:.3f} | PR-AUC {p['pr_auc']:.3f} (random {res['test']['random_pr_auc']:.3f}) | "
          f"precision {p['precision']:.3f} | recall {p['recall']:.3f} @ clean-val 99th-percentile threshold")
    for k in ("scale_ablation", "source_ablation", "scale_attention_mean"):
        if k in res["test"]: print(k, json.dumps(res["test"][k], indent=1))
else:
    R = json.load(open(f"{OUT}/results.json"))
    print(json.dumps({k: v for k, v in R.items() if k not in ("details", "provenance")}, indent=1)[:4000])

In [ ]:
# Ensemble only: each branch alone vs fused (first seed)
if MODE == "ensemble":
    d = R["details"][str(SEEDS[0])]
    g = lambda mm: mm.get("pr_auc", float("nan"))
    names = list(d["branch_alone"]) + ["FUSED"]
    plt.figure(figsize=(8, 3.5)); plt.barh(names, [g(d["branch_alone"][b]) for b in d["branch_alone"]] + [g(d["main"])])
    plt.title(f"point-wise PR-AUC, seed {SEEDS[0]}"); plt.show()
    print("fusion variants (rule, dropped branch, PR-AUC):")
    for v in d["fusion_variants"]: print("  ", v["rule"], v.get("drop_branch"), round(g(v["metrics"]), 3))

## 9. What the model sees on individual windows (single mode)
Test-year windows with injected errors: left = clean (gray) vs injected (red); right = per-day fused anomaly score vs the calibrated threshold.
A good detector's score rises on the red days.

In [ ]:
if MODE == "single":
    ts = m.test_s; ts.rng = np.random.default_rng(5)
    b = ts.batch(512, inject_rate=1.0)
    fused = m.cal.fuse(m.trainer.score(b))
    idx = [i for i in range(len(b.kind)) if b.kind[i] >= 0][:6]
    fig, axes = plt.subplots(len(idx), 2, figsize=(15, 2.6 * len(idx)))
    for row, i in zip(axes, idx):
        row[0].bar(range(b.x_mm.shape[1]), b.clean_mm[i], color="lightgray", width=1.0)
        row[0].bar(np.where(b.labels[i])[0], b.x_mm[i, :, 0][b.labels[i]], color="crimson", width=1.0)
        row[0].set_title(f"kind: {KINDS[b.kind[i]]}", fontsize=9)
        row[1].plot(fused[i], color="k"); row[1].axhline(m.cal.threshold, color="orange", ls="--", label="threshold")
        row[1].fill_between(range(fused.shape[1]), 0, max(float(fused[i].max()), 1e-6), where=b.labels[i], color="crimson", alpha=0.2)
        row[1].set_title("fused anomaly score", fontsize=9)
    axes[0, 1].legend(); plt.tight_layout(); plt.show()

## 10. Save / download
Everything is in the output directory (`manifest.json` + `model.pt` for single mode; `results.json` + `seed_*/` for ensemble).
On Kaggle it appears in the notebook's **Output** tab; the zip below makes it one download. Load a trained model later with
`from training import artifact; artifact.load("<dir>")`.

In [ ]:
import shutil
zip_path = shutil.make_archive(str(pathlib.Path(OUT).parent / f"{RUN_NAME}_artifacts"), "zip", OUT)
print("zipped:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")